# Dataset Builder


### Data Dictionary

| Column | Description |
|:--|:--|
| **date** | Trading date |
| **ticker** | Asset symbol (e.g., SPY, TLT, GLD, QQQ) |
| **logvol_t** | log of daily realized volatility (base proxy, default Garman–Klass) |
| **target_logvol_t+1**, **+5**, **+22** | Future log-volatility at 1-, 5-, 22-day horizons — supervised targets |
| **logvol_t-1**, **-5**, **-22** | Lagged log-volatility features |
| **ret_1d** | 1-day log return (close-to-close) |
| **roll_mean_ret_Xd**, **roll_std_ret_Xd** | Rolling mean/std of returns over X days |
| **roll_mean_vol_Xd**, **roll_std_vol_Xd** | Rolling mean/std of realized vol (daily proxy) over X days |
| **vix**, **vix3m** | CBOE VIX and 3-month VIX (VIX3M) indices |
| **vix_term_spread** | VIX3M − VIX |
| **weekday** | Day of week (0 = Mon … 4 = Fri) |
| **is_month_end**, **is_opex**, **is_holiday_eve** | Calendar flags for month-end, 3rd Friday options expiry, and holiday-adjacent days |
| **dow_sin**, **dow_cos** | Cyclical encodings of weekday |
| **cross_vol_rank** | Cross-sectional rank of daily vol across assets (shifted to t−1 per ticker) |
| **pca_vol_cs_i** | i-th principal component from cross-asset vol (shifted to t−1) |


## Imports & Parameters

In [1]:

import os, math, numpy as np, pandas as pd, yfinance as yf
from typing import List, Optional
from sklearn.decomposition import PCA

# ---- User-editable ----
INPUT_CSV = "realized_vol_proxies.csv"   # output of the RV proxies notebook
PRIMARY_VOL_BASE = "gk"                  # one of: "gk", "parkinson", "rs", "cc"
USE_ANNUALIZED = False                   # if True, use *_ann columns
HORIZONS: List[int] = [1, 5, 22]
ROLL_WINDOWS = [5, 22, 66, 132, 252]     # rolling windows for features
ADD_PCA = True
N_PCA = 3

# Splits & embargo
TRAIN_START = "2010-01-01"; TRAIN_END = "2017-12-31"
VAL_START   = "2018-01-01"; VAL_END   = "2019-12-31"
TEST_START  = "2020-01-01"; TEST_END  = "2024-12-31"
EMBARGO_DAYS = 5

# Outputs
OUT_FULL  = "tft_ready_dataset.csv"
OUT_TRAIN = "tft_ready_train.csv"
OUT_VAL   = "tft_ready_val.csv"
OUT_TEST  = "tft_ready_test.csv"


## Calendar helpers

In [2]:

def is_third_friday(d: pd.Timestamp) -> bool:
    if d.weekday() != 4:  # Friday
        return False
    first = d.replace(day=1)
    first_friday = first + pd.offsets.Week(weekday=4)
    third_friday = first_friday + pd.offsets.Week(2)
    return d == third_friday

def build_calendar_flags(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["weekday"] = out["date"].dt.weekday
    out["is_month_end"] = out["date"].dt.is_month_end.astype(int)
    out["is_opex"] = out["date"].apply(lambda d: int(is_third_friday(pd.Timestamp(d))))
    # Simple "holiday-eve" proxy: Fridays and month-ends
    out["is_holiday_eve"] = ((out["weekday"] == 4) | (out["is_month_end"] == 1)).astype(int)
    # Cyclical encodings
    out["dow_sin"] = np.sin(2 * math.pi * out["weekday"] / 5.0)
    out["dow_cos"] = np.cos(2 * math.pi * out["weekday"] / 5.0)
    return out


## Load realized-volatility proxies

In [3]:

raw = pd.read_csv(INPUT_CSV, parse_dates=["date"]).sort_values(["ticker","date"]).reset_index(drop=True)
print(raw.head(3)); print("\nTickers:", sorted(raw["ticker"].unique())); print("Date range:", raw["date"].min(), "->", raw["date"].max())


        date ticker        open        high         low       close  \
0 2010-01-04    GLD  109.820000  110.139999  109.309998  109.800003   
1 2010-01-05    GLD  109.879997  110.389999  109.260002  109.699997   
2 2010-01-06    GLD  110.709999  111.769997  110.410004  111.510002   

    adj_close    volume   log_ret        var_cc  ...  vol_gk_5d_ann  \
0  109.800003  16224100       NaN           NaN  ...            NaN   
1  109.699997  14213100 -0.000911  8.303172e-07  ...            NaN   
2  111.510002  24981900  0.016365  2.678117e-04  ...            NaN   

   vol_rs_5d_ann  vol_cc_22d  vol_parkinson_22d  vol_gk_22d  vol_rs_22d  \
0            NaN         NaN                NaN         NaN         NaN   
1            NaN         NaN                NaN         NaN         NaN   
2            NaN         NaN                NaN         NaN         NaN   

   vol_cc_22d_ann  vol_parkinson_22d_ann  vol_gk_22d_ann  vol_rs_22d_ann  
0             NaN                    NaN             N

## Targets (t+H) and lag features

In [4]:

def vol_col_name(base: str, H: Optional[int]=None, annualized: bool=False) -> str:
    pre = f"vol_{base.lower()}"
    if H is None or H == 1:
        return pre if not annualized else f"{pre}_ann"
    return f"{pre}_{H}d" if not annualized else f"{pre}_{H}d_ann"

df = raw.copy()
primary_cols = {}
for H in HORIZONS:
    cname = vol_col_name(PRIMARY_VOL_BASE, H=H, annualized=USE_ANNUALIZED)
    if cname not in df.columns:
        raise ValueError(f"Missing expected column: {cname}. Check PRIMARY_VOL_BASE/USE_ANNUALIZED.")
    primary_cols[H] = cname

# Targets: log-vol at t+H
for H in HORIZONS:
    df[f"target_logvol_t+{H}"] = np.log(df.groupby("ticker")[primary_cols[H]].shift(-H))

# Today's log-vol and lagged features
df["logvol_t"] = np.log(df[primary_cols[1]])
for lag in [1, 5, 22]:
    df[f"logvol_t-{lag}"] = df.groupby("ticker")["logvol_t"].shift(lag)


## Returns and rolling statistics (leakage-safe)

In [5]:

# 1-day log return
df["ret_1d"] = np.log(df.groupby("ticker")["close"].shift(0) / df.groupby("ticker")["close"].shift(1))

# Rolling stats on returns and on the primary daily vol column, then shift by 1 day
base_vol = primary_cols[1]
for win in ROLL_WINDOWS:
    df[f"roll_mean_ret_{win}d"] = df.groupby("ticker")["ret_1d"].rolling(win, min_periods=win).mean().reset_index(level=0, drop=True)
    df[f"roll_std_ret_{win}d"]  = df.groupby("ticker")["ret_1d"].rolling(win, min_periods=win).std().reset_index(level=0, drop=True)
    df[f"roll_mean_vol_{win}d"] = df.groupby("ticker")[base_vol].rolling(win, min_periods=win).mean().reset_index(level=0, drop=True)
    df[f"roll_std_vol_{win}d"]  = df.groupby("ticker")[base_vol].rolling(win, min_periods=win).std().reset_index(level=0, drop=True)

# Shift engineered features by 1 day to ensure features at t use info <= t-1
feature_cols = [c for c in df.columns if c.startswith("roll_")] + ["ret_1d"] + [f"logvol_t-{lag}" for lag in [1,5,22]]
df[feature_cols] = df.groupby("ticker")[feature_cols].shift(1)


## Exogenous: VIX and VIX3M (robust fetch & merge)

In [6]:

def _normalize_cols(cols):
    out = []
    for c in cols:
        if isinstance(c, tuple):
            c = "_".join([str(x) for x in c if x is not None and str(x) != ""])
        else:
            c = str(c)
        out.append(c.lower().replace(" ", "_"))
    return out

def fetch_index_series(ticker: str, start: str, end: Optional[str]=None) -> pd.DataFrame:
    """Return (date, close) with robust handling for MultiIndex/tuple columns and schema drift."""
    data = yf.download(
        ticker, start=start, end=end,
        interval="1d", auto_adjust=False, progress=False, group_by=None
    )
    if data is None or len(data) == 0:
        raise ValueError(f"No data returned for {ticker}")
    data = data.reset_index()
    data.columns = _normalize_cols(data.columns)

    # Prefer adj_close; fallback to close; else any *_adj_close or *_close
    close_col = None
    if "adj_close" in data.columns: close_col = "adj_close"
    elif "close" in data.columns:  close_col = "close"
    else:
        candidates = [c for c in data.columns if c.endswith("_adj_close")] or                      [c for c in data.columns if c.endswith("_close")]
        if candidates: close_col = candidates[0]
    if close_col is None:
        raise KeyError(f"No close/adj_close column for {ticker}. Columns: {data.columns}")

    df_idx = data[["date", close_col]].rename(columns={close_col: "close"}).copy()
    df_idx["date"] = pd.to_datetime(df_idx["date"]).dt.tz_localize(None)
    return df_idx

start_all = str(df["date"].min().date())
vix   = fetch_index_series("^VIX",   start=start_all, end=None)
vix3m = fetch_index_series("^VIX3M", start=start_all, end=None)

exo = (
    pd.merge(vix, vix3m, on="date", how="outer", suffixes=("_vix", "_vix3m"))
      .sort_values("date")
      .ffill()
)

# Expect 'close_vix' and 'close_vix3m'; otherwise detect safely
vix_col   = "close_vix"   if "close_vix"   in exo.columns else next((c for c in exo.columns if c.startswith("close") and "vix3m" not in c), None)
vix3m_col = "close_vix3m" if "close_vix3m" in exo.columns else next((c for c in exo.columns if c.startswith("close") and "vix3m" in c), None)
if vix_col is None or vix3m_col is None:
    # Fallback: find any with substrings
    vix_col   = vix_col   or exo.filter(like="vix").columns[-1]
    vix3m_col = vix3m_col or exo.filter(like="vix3m").columns[-1]

exo["vix"] = exo[vix_col]
exo["vix3m"] = exo[vix3m_col]
exo["vix_term_spread"] = exo["vix3m"] - exo["vix"]
exo = exo[["date","vix","vix3m","vix_term_spread"]]

# Merge into main df and shift one day to avoid leakage
df = (
    pd.merge(df, exo, on="date", how="left")
      .sort_values(["ticker", "date"])
      .reset_index(drop=True)
)
df[["vix", "vix3m", "vix_term_spread"]] = (
    df.groupby("ticker")[["vix", "vix3m", "vix_term_spread"]].ffill()
)
df[["vix", "vix3m", "vix_term_spread"]] = (
    df.groupby("ticker")[["vix", "vix3m", "vix_term_spread"]].shift(1)
)


## Calendar flags & cyclical encodings

In [7]:

df = build_calendar_flags(df)


## Cross-asset features: ranks & PCA

In [8]:

# Rank today's base vol across tickers (then shift t-1 per ticker)
base_vol = primary_cols[1]
df["cross_vol_rank"] = df.groupby("date")[base_vol].rank(method="average")
df["cross_vol_rank"] = df.groupby("ticker")["cross_vol_rank"].shift(1)

if ADD_PCA:
    pivot = df.pivot_table(index="date", columns="ticker", values=base_vol)
    pct = (pivot - pivot.mean(axis=1).values.reshape(-1,1)) / (pivot.std(axis=1).replace(0, np.nan).values.reshape(-1,1))
    pct = pct.fillna(0.0)
    pca = PCA(n_components=min(N_PCA, pct.shape[1]))
    comps = pca.fit_transform(pct.values)
    comp_cols = [f"pca_vol_cs_{i+1}" for i in range(comps.shape[1])]
    pca_df = pd.DataFrame(comps, index=pct.index, columns=comp_cols).reset_index().rename(columns={"index":"date"})
    df = pd.merge(df, pca_df, on="date", how="left").sort_values(["ticker","date"]).reset_index(drop=True)
    df[comp_cols] = df.groupby("ticker")[comp_cols].shift(1)


## Finalize feature set, split, and save

In [9]:

target_cols = [f"target_logvol_t+{H}" for H in HORIZONS]
base_feats = ["weekday","is_month_end","is_opex","is_holiday_eve","dow_sin","dow_cos",
              "vix","vix3m","vix_term_spread",
              "ret_1d","cross_vol_rank",
              "logvol_t-1","logvol_t-5","logvol_t-22"]
roll_feats = [c for c in df.columns if c.startswith("roll_")]
pca_feats = [c for c in df.columns if c.startswith("pca_vol_cs_")]

keep = ["date","ticker","logvol_t"] + target_cols + base_feats + roll_feats + pca_feats
panel = df[keep].dropna().sort_values(["ticker","date"]).reset_index(drop=True)

def trim_with_embargo(df, start, end, embargo):
    m = (df["date"] >= pd.Timestamp(start)) & (df["date"] <= pd.Timestamp(end))
    sub = df[m].copy()
    def trim_grp(g):
        if len(g) <= 2*embargo: return g.iloc[0:0]
        return g.iloc[embargo: -embargo]
    return sub.groupby("ticker", group_keys=False).apply(trim_grp)

train_df = trim_with_embargo(panel, TRAIN_START, TRAIN_END, EMBARGO_DAYS)
val_df   = trim_with_embargo(panel, VAL_START, VAL_END, EMBARGO_DAYS)
test_df  = trim_with_embargo(panel, TEST_START, TEST_END, EMBARGO_DAYS)

panel.to_csv(OUT_FULL, index=False)
train_df.to_csv(OUT_TRAIN, index=False)
val_df.to_csv(OUT_VAL, index=False)
test_df.to_csv(OUT_TEST, index=False)

print("Saved files:\n ", OUT_FULL, "\n ", OUT_TRAIN, "\n ", OUT_VAL, "\n ", OUT_TEST)


/var/folders/py/0h47_k_94pb61f_4bbkqkvlh0000gn/T/ipykernel_12674/2644720905.py:18: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return sub.groupby("ticker", group_keys=False).apply(trim_grp)
/var/folders/py/0h47_k_94pb61f_4bbkqkvlh0000gn/T/ipykernel_12674/2644720905.py:18: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return sub.groupby("ticker", group_keys=False).apply(trim_grp)
/var/folders/py/0h47_k_94p

Saved files:
  tft_ready_dataset.csv 
  tft_ready_train.csv 
  tft_ready_val.csv 
  tft_ready_test.csv
